# Análise da base canônica e identidades processuais

Exploração read-only. As extrações abaixo são experimentais e não constituem índice ou resolvedor.

# 1. Setup

In [ ]:
from __future__ import annotations

import hashlib
import platform
import re
from collections import Counter

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from bracis_jusbrasil.database import connect_database, get_database_path

DATABASE_PATH = get_database_path()
connection = connect_database(DATABASE_PATH)
WINDOWS = (500, 1_000, 2_500, 5_000, 10_000)
TRIBUNAIS = ('STF', 'STJ', 'TSE', 'TST', 'STM')
print({'python': platform.python_version(), 'database': str(DATABASE_PATH)})

In [ ]:
def query_frame(sql: str, params: tuple[object, ...] = ()) -> pd.DataFrame:
    return pd.read_sql_query(sql, connection, params=params)


def preview(text: str, limit: int = 360) -> str:
    compact = ' '.join(text.split())
    return compact[:limit] + ('…' if len(compact) > limit else '')


def normalize_number(value: str | None) -> str | None:
    if value is None:
        return None
    digits = re.sub(r'[^0-9]', '', value)
    return digits or None


def normalize_class(value: str | None) -> str | None:
    if not value:
        return None
    normalized = re.sub(r'[^A-Z0-9]+', ' ', value.upper()).strip()
    return normalized or None

# 2. Universo analisado

In [ ]:
acordaos = query_frame(
    "SELECT rowid, documento_id, id, tribunal, ano, relator, texto, texto_len FROM documentos WHERE natureza = 'acordao' ORDER BY documento_id"
)
print('acórdãos:', len(acordaos))
display(acordaos[['documento_id', 'id', 'tribunal', 'ano', 'relator', 'texto_len']].head())

por_tribunal = (acordaos.groupby('tribunal', as_index=False)
    .agg(registros=('id', 'size'), primeiro_ano=('ano', 'min'), ultimo_ano=('ano', 'max'),
         texto_medio=('texto_len', 'mean'), texto_minimo=('texto_len', 'min'), texto_maximo=('texto_len', 'max'),
         relator_nulo=('relator', lambda values: int(values.isna().sum())))
    .sort_values('tribunal'))
display(por_tribunal)
display(acordaos[['tribunal', 'ano', 'relator', 'texto_len']].isna().sum().rename('nulos').to_frame())

# 3. Anatomia por tribunal

Amostras mostram que a identidade aparece em posições e formatos diferentes; a tabela é observacional.

In [ ]:
amostras = (acordaos.groupby('tribunal', group_keys=False).head(2)
    [['tribunal', 'documento_id', 'id', 'relator', 'texto']].copy())
amostras['inicio_relevante'] = amostras['texto'].map(lambda value: preview(value, 900))
display(amostras.drop(columns='texto'))

anatomia = pd.DataFrame([
    ('STF', 'início, após data/turma', 'decimal curto', 'forma abreviada ou extensa', 'estado por extenso frequente', 'número do feito pode coexistir com referências no corpo'),
    ('STJ', 'início do documento', 'decimal e sufixo de UF', 'título antes de Nº', 'sigla após hífen', 'número interno também pode aparecer'),
    ('TSE', 'início do acórdão', 'CNJ eleitoral', 'título extenso', 'município e estado', 'classe eleitoral acompanha número'),
    ('TST', 'após ementa em parte dos casos', 'CNJ trabalhista', 'siglas compostas', 'embutida no CNJ', 'posição é mais tardia'),
    ('STM', 'início, após cabeçalho da sessão', 'CNJ militar', 'título extenso', 'sufixo /UF', 'ata e acórdão convivem no texto'),
], columns=['tribunal', 'localização típica', 'número', 'classe', 'UF', 'observações'])
display(anatomia)

# 4. Posição da identidade

In [ ]:
CNJ_RE = re.compile(r'(?<!\d)\d{3,7}-\d{2}\.\d{4}\.\d\.\d{2}\.\d{4}(?:/[A-Z]{2})?(?!\d)')
DECIMAL_RE = re.compile(r'(?<!\d)\d{1,3}(?:[.,]\d{3})+(?:-\d+)?(?!\d)')
UF_RE = re.compile(r'(?:[-/]\s*|\b)(AC|AL|AP|AM|BA|CE|DF|ES|GO|MA|MT|MS|MG|PA|PB|PR|PE|PI|RJ|RN|RS|RO|RR|SC|SP|SE|TO)\b')
STATE_TO_UF = {'PARANÁ':'PR', 'SÃO PAULO':'SP', 'RIO DE JANEIRO':'RJ', 'DISTRITO FEDERAL':'DF', 'BAHIA':'BA', 'MINAS GERAIS':'MG', 'PERNAMBUCO':'PE', 'CEARÁ':'CE', 'RIO GRANDE DO SUL':'RS'}


def class_before_number(text: str, position: int) -> str | None:
    prefix = text[max(0, position - 150):position].replace('\n', ' ')
    chunks = re.findall(r'[A-ZÀ-Ú][A-ZÀ-Ú .-]{2,}', prefix.upper())
    if not chunks:
        return None
    candidate = chunks[-1].strip(' -.')
    return candidate[-100:] if len(candidate) >= 3 else None


def experimental_identity(row: pd.Series) -> dict[str, object]:
    """Extrai somente forma canônica visível; não corrige OCR nem resolve feito."""
    text = row['texto']
    tribunal = row['tribunal']
    search_limit = 10_000 if tribunal in {'STJ', 'TST'} else 2_500
    window = text[:search_limit]
    match = None
    if tribunal in {'TSE', 'TST', 'STM'}:
        match = CNJ_RE.search(window)
    if match is None:
        match = DECIMAL_RE.search(window)
    if match is None and tribunal == 'STF':
        match = CNJ_RE.search(window)
    numero_raw = match.group(0) if match else None
    position = match.start() if match else None
    classe_raw = class_before_number(window, position) if position is not None else None
    after = window[position:position + 180] if position is not None else ''
    uf_match = UF_RE.search(after)
    uf = uf_match.group(1) if uf_match else None
    if uf is None:
        upper = after.upper()
        uf = next((code for state, code in STATE_TO_UF.items() if state in upper), None)
    quality = 'estrutural' if numero_raw and classe_raw else ('parcial' if numero_raw else 'sem_numero')
    return {'classe_raw': classe_raw, 'classe_normalizada': normalize_class(classe_raw),
            'numero_raw': numero_raw, 'numero_normalizado': normalize_number(numero_raw),
            'uf': uf, 'match_position': position, 'janela_analisada': search_limit, 'qualidade': quality}

parsed = acordaos.join(pd.DataFrame([experimental_identity(row) for _, row in acordaos.iterrows()]))
print('Identidade mínima experimental: tribunal + numero_normalizado.')

In [ ]:
posicoes = parsed.groupby('tribunal', as_index=False).agg(
    mediana_posicao=('match_position', 'median'),
    p90_posicao=('match_position', lambda values: values.dropna().quantile(.9)),
    encontrados=('numero_normalizado', lambda values: int(values.notna().sum()))
)
for threshold in WINDOWS:
    posicoes[f'até_{threshold}'] = parsed.groupby('tribunal')['match_position'].apply(lambda values: int((values <= threshold).sum())).values
display(posicoes)

fig, ax = plt.subplots(figsize=(8, 4))
parsed.dropna(subset=['match_position']).boxplot(column='match_position', by='tribunal', ax=ax)
plt.suptitle('')
ax.set_title('Distribuição da posição do provável número')
ax.set_ylabel('posição no texto')
plt.show()

In [ ]:
posicoes_acumuladas = posicoes.melt(
    id_vars=['tribunal', 'encontrados'],
    value_vars=[f'até_{threshold}' for threshold in WINDOWS],
    var_name='janela', value_name='encontrados_na_janela',
)
posicoes_acumuladas['limite'] = posicoes_acumuladas['janela'].str.removeprefix('até_').astype(int)
posicoes_acumuladas['cobertura_percentual'] = (
    100 * posicoes_acumuladas['encontrados_na_janela'] / posicoes_acumuladas['encontrados']
)
fig, ax = plt.subplots(figsize=(8, 4))
for tribunal, grupo in posicoes_acumuladas.groupby('tribunal'):
    grupo = grupo.sort_values('limite')
    ax.plot(grupo['limite'], grupo['cobertura_percentual'], marker='o', label=tribunal)
ax.set_title('Cobertura acumulada por posição no texto')
ax.set_xlabel('limite da janela (caracteres)')
ax.set_ylabel('% dos números encontrados')
ax.set_ylim(0, 105)
ax.legend(title='tribunal')
ax.grid(axis='y', alpha=.25)
fig.tight_layout()
plt.show()

# 5. Parsing exploratório

O parser conserva `classe_raw` e `numero_raw`; a normalização remove somente pontuação para facilitar comparação experimental.

In [ ]:
display(parsed[['tribunal', 'documento_id', 'classe_raw', 'classe_normalizada', 'numero_raw', 'numero_normalizado', 'uf', 'match_position', 'qualidade']].groupby('tribunal', group_keys=False).head(3))

# 6. Métricas de cobertura

In [ ]:
cobertura = (parsed.groupby('tribunal', as_index=False)
    .agg(registros=('id', 'size'), numero_extraido=('numero_normalizado', lambda values: int(values.notna().sum())),
         classe_extraida=('classe_normalizada', lambda values: int(values.notna().sum())),
         uf_extraida=('uf', lambda values: int(values.notna().sum())),
         identidade_minima=('numero_normalizado', lambda values: int(values.notna().sum())))
    .sort_values('tribunal'))
cobertura.loc['Total'] = ['Total', len(parsed), *[int(cobertura[column].sum()) for column in cobertura.columns[2:]]]
display(cobertura)
print('Identidade mínima = tribunal + numero_normalizado; não é uma decisão de chave final.')

In [ ]:
cobertura_visual = cobertura.query("tribunal != 'Total'").set_index('tribunal')
cobertura_visual = 100 * cobertura_visual[['numero_extraido', 'classe_extraida', 'uf_extraida']].div(cobertura_visual['registros'], axis=0)
cobertura_visual = cobertura_visual.rename(columns={'numero_extraido': 'número', 'classe_extraida': 'classe', 'uf_extraida': 'UF'})
ax = cobertura_visual.plot.bar(figsize=(8, 4), ylim=(0, 105), rot=0, color=['#4472c4', '#70ad47', '#ed7d31'])
ax.set_title('Cobertura das extrações experimentais')
ax.set_xlabel('tribunal')
ax.set_ylabel('% dos acórdãos')
ax.legend(title='campo')
ax.grid(axis='y', alpha=.25)
ax.figure.tight_layout()
plt.show()

# 7. Inspeção das falhas

In [ ]:
def failure_category(row: pd.Series) -> str:
    if pd.isna(row['numero_normalizado']):
        return 'número não localizado na janela experimental'
    if pd.isna(row['classe_normalizada']):
        return 'classe não isolada pelo padrão simples'
    if row['match_position'] > 2_500:
        return 'identidade tardia (STJ/TST)'
    return 'parse estrutural mínimo'

parsed['categoria'] = parsed.apply(failure_category, axis=1)
taxonomia = parsed.groupby(['tribunal', 'categoria'], as_index=False).size().sort_values(['tribunal', 'size'], ascending=[True, False])
display(taxonomia)
falhas = parsed[parsed['categoria'] != 'parse estrutural mínimo'].copy()
falhas['trecho'] = falhas['texto'].map(lambda value: preview(value, 550))
display(falhas[['tribunal', 'documento_id', 'categoria', 'match_position', 'trecho']].head(15))

In [ ]:
ax = (taxonomia.pivot(index='tribunal', columns='categoria', values='size').fillna(0)
    .plot.bar(stacked=True, figsize=(9, 4), rot=0))
ax.set_title('Categorias do parsing exploratório')
ax.set_xlabel('tribunal')
ax.set_ylabel('acórdãos')
ax.legend(title='categoria', bbox_to_anchor=(1.02, 1), loc='upper left')
ax.figure.tight_layout()
plt.show()

# 8. Classes processuais

In [ ]:
classes = (parsed.dropna(subset=['classe_raw']).groupby(['tribunal', 'classe_raw', 'classe_normalizada'], as_index=False)
    .agg(frequencia=('id', 'size'), exemplos=('documento_id', lambda values: ', '.join(map(str, values.head(3)))))
    .sort_values(['tribunal', 'frequencia'], ascending=[True, False]))
display(classes.groupby('tribunal', group_keys=False).head(12))
print('Normalização conceitual: preservar a forma bruta e comparar apenas caixa/pontuação em experimentos.')

# 9. Números processuais

In [ ]:
def number_family(row: pd.Series) -> str:
    value = row['numero_raw'] if isinstance(row['numero_raw'], str) else ''
    if CNJ_RE.fullmatch(value):
        return 'CNJ com possível UF'
    if row['tribunal'] == 'STF':
        return 'convencional STF'
    if row['tribunal'] == 'STJ':
        return 'convencional STJ'
    return 'outro formato observado'

parsed['familia_numero'] = parsed.apply(number_family, axis=1)
numeros = parsed.groupby(['tribunal', 'familia_numero'], as_index=False).agg(frequencia=('id', 'size'), exemplos=('numero_raw', lambda values: ', '.join(map(str, values.dropna().head(4)))))
display(numeros)
print('Estratégia segura: manter o bruto e normalizar somente para dígitos; não há correção OCR nesta análise canônica.')

# 10. UF e estado

In [ ]:
uf_metricas = (parsed.groupby('tribunal', as_index=False)
    .agg(uf_direta=('uf', lambda values: int(values.notna().sum())),
         ausencia_uf=('uf', lambda values: int(values.isna().sum())),
         exemplos_uf=('uf', lambda values: ', '.join(sorted(set(values.dropna()))[:8]))))
display(uf_metricas)
print('A medição não pressupõe que UF seja obrigatória: ela será comparada nas chaves candidatas.')

# 11. Duplicatas e múltiplos registros

In [ ]:
parsed['texto_hash'] = parsed['texto'].map(lambda value: hashlib.sha256(value.encode('utf-8')).hexdigest())
textos_duplicados = parsed[parsed['texto_hash'].duplicated(keep=False)]
print({'grupos_texto_identico': int(textos_duplicados['texto_hash'].nunique()), 'registros_envolvidos': len(textos_duplicados)})

identity_base = parsed.dropna(subset=['numero_normalizado']).groupby(
    ['tribunal', 'classe_normalizada', 'numero_normalizado', 'uf'], dropna=False, as_index=False
)
mesma_identidade = identity_base.agg(
    quantidade_ids=('id', 'nunique'),
    ids=('id', lambda values: ', '.join(sorted(set(map(str, values))))),
    textos_identicos=('texto_hash', lambda values: values.nunique() == 1),
)
mesma_identidade = mesma_identidade.query('quantidade_ids > 1').sort_values('quantidade_ids', ascending=False)
display(mesma_identidade.head(20))

In [ ]:
tamanho_grupos = mesma_identidade['quantidade_ids'].value_counts().sort_index()
ax = tamanho_grupos.plot.bar(figsize=(7, 4), rot=0, color='#5b9bd5')
ax.set_title('Tamanho dos grupos de identidade experimental')
ax.set_xlabel('IDs distintos por grupo')
ax.set_ylabel('quantidade de grupos')
ax.figure.tight_layout()
plt.show()

# 12. Chaves candidatas de feito

In [ ]:
candidate_rows = parsed.dropna(subset=['numero_normalizado']).copy()
candidate_rows['chave_a'] = candidate_rows['tribunal'] + '|' + candidate_rows['numero_normalizado']
candidate_rows['chave_b'] = candidate_rows['chave_a'] + '|' + candidate_rows['classe_normalizada'].fillna('<sem_classe>')
candidate_rows['chave_c'] = candidate_rows['chave_b'] + '|' + candidate_rows['uf'].fillna('<sem_uf>')

key_rows = []
for name in ('chave_a', 'chave_b', 'chave_c'):
    grouped = candidate_rows.groupby(name)['id'].nunique()
    collisions = grouped[grouped > 1]
    key_rows.append({'chave': name, 'grupos_unicos': int(grouped.size), 'colisoes': int(collisions.size), 'grupos_multi_id': int(collisions.size), 'maior_grupo_ids': int(grouped.max())})
display(pd.DataFrame(key_rows))

for name in ('chave_a', 'chave_b', 'chave_c'):
    examples = candidate_rows.groupby(name).filter(lambda group: group['id'].nunique() > 1)
    if not examples.empty:
        print(name)
        display(examples[['tribunal', 'classe_raw', 'numero_raw', 'uf', 'id', 'documento_id']].head(10))

In [ ]:
resumo_chaves = pd.DataFrame(key_rows)
ax = resumo_chaves.plot.bar(x='chave', y=['grupos_unicos', 'colisoes'], figsize=(7, 4), rot=0, color=['#4472c4', '#c55a11'])
ax.set_title('Colisões nas chaves candidatas')
ax.set_xlabel('composição experimental')
ax.set_ylabel('quantidade de grupos')
ax.legend(['grupos únicos', 'colisões'])
ax.figure.tight_layout()
plt.show()

# 13. FTS como fallback

FTS é medido como recuperação textual: não escolhe um ID e não substitui uma identidade estruturada.

In [ ]:
def fts_example(query: str, occurrence: str, limit: int = 10) -> pd.DataFrame:
    rows = query_frame(
        '''SELECT d.documento_id, d.id, d.tribunal, d.texto
           FROM documentos_fts JOIN documentos AS d ON d.rowid = documentos_fts.rowid
           WHERE documentos_fts MATCH ? LIMIT ?''', (query, limit))
    rows['posição'] = rows['texto'].map(lambda value: value.casefold().find(occurrence.casefold()))
    rows['início'] = rows['texto'].map(lambda value: preview(value, 260))
    rows['contexto'] = rows.apply(lambda row: preview(row['texto'][max(0, row['posição'] - 80):row['posição'] + 180], 280), axis=1)
    return rows.drop(columns='texto')

display(fts_example('"1.276.977"', '1.276.977'))
print('Posições menores e contextos de cabeçalho são indícios exploratórios; FTS deve ser fallback de recuperação.')

# 14. Findings

## Fatos observados

- Os cinco tribunais apresentam posições e superfícies de identificação distintas.
- Texto idêntico e múltiplos IDs existem no acervo e precisam ser medidos separadamente.
- FTS recupera menções no corpo e não é, sozinho, uma chave de feito.

## Hipóteses de arquitetura

- Uma futura identidade deve preservar tribunal, número bruto/normalizado, classe e UF separadamente.
- Parsers por tribunal parecem necessários; classe e UF devem ser avaliadas como componentes de desambiguação, não pressupostos universais.

## Questões abertas

- Quais colisoes representam o mesmo feito jurídico e quais refletem documentos distintos?
- Qual cobertura experimental é suficiente para transformar estes padrões em uma implementação de produção?

In [ ]:
connection.close()
print('Conexão read-only encerrada.')